# docstring-tuner — QLoRA training on a Colab T4

**Set the runtime to a GPU (T4):** Runtime -> Change runtime type -> T4 GPU.

This notebook builds the dataset, trains the LoRA adapter, and generates base vs
fine-tuned docstrings over the held-out test set. Download the results at the end and
run the LLM-judge evaluation locally (that step uses your local `claude` CLI).

In [ ]:
!nvidia-smi

## 1. Get the code

The repo is private, so authenticate the clone with a GitHub personal access token
(Settings -> Developer settings -> Fine-grained tokens, read access to the repo).
Paste it below, or make the repo public first and use the plain URL.

In [ ]:
GITHUB_USER = 'paffon'
REPO = 'docstring-tuner'
TOKEN = ''  # <-- paste a fine-grained PAT with read access (leave blank if public)
url = f'https://{TOKEN + "@" if TOKEN else ""}github.com/{GITHUB_USER}/{REPO}.git'
!git clone $url

In [ ]:
%cd docstring-tuner
!pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps

## 2. Build the dataset

Streams a slice of CodeSearchNet, strips docstrings, and writes disjoint
`data/train.jsonl` / `data/test.jsonl`. Takes a few minutes on first run.

In [ ]:
!python -m docstring_tuner.data

## 3. Train the QLoRA adapter

4-bit QLoRA SFT. On a T4 this is ~5-10 minutes for the default config. The adapter
(only) is saved to `artifacts/adapter/`.

In [ ]:
!python -m docstring_tuner.train

## 4. Generate base vs fine-tuned docstrings over the test set

This writes `artifacts/generations.jsonl` and prints a quick comparison using the
offline **mock** judge (Colab has no `claude` CLI). The real LLM-judge scoring happens
locally after you download the results.

In [ ]:
!python -m docstring_tuner.evaluate --generate --judge mock --limit 150

## 5. Download the adapter + generations

In [ ]:
!zip -r artifacts.zip artifacts/adapter artifacts/generations.jsonl
from google.colab import files
files.download('artifacts.zip')

## 6. Back on your machine

```powershell
# unzip artifacts.zip into the repo so you have artifacts/adapter and
# artifacts/generations.jsonl, then:
python -m docstring_tuner.evaluate   # scores base vs tuned with your local claude judge
python -m docstring_tuner.demo --limit 3
```